# Regional Calibration Diagnostics for the PEcAn Carbon Reanalysis

This notebook breaks the ensemble calibration assessment of the PEcAn state data
assimilation (SDA) carbon reanalysis down by region, rather than reporting a
single continent-wide number. It asks: **does the trustworthiness of the
ensemble spread vary by biome and by ecoregion?**

We evaluate three carbon-cycle variables (aboveground biomass, soil carbon, and
leaf area index) at roughly 8000 SDA sites across North America, comparing the
downscaled ensemble against independent observational benchmarks, and grouping
the calibration diagnostics two ways:

1. by **land cover class** (biome), and
2. by **EPA/CEC ecoregion** (Level 1 and Level 2).

The calibration diagnostics themselves come from `ensemble_calibration.py`, so
every per-region number is computed exactly the same way as the domain-wide
ones. This notebook orchestrates the regional breakdown using the helper modules
`regional_diagnostics.py`, `ecoregion_join.py`, and `regional_figures.py`.


## Setup

Load the site coordinates and land cover classes, and sample the ensemble and
benchmark at each site once (reused throughout).

In [ ]:
import numpy as np
import ensemble_calibration as ec
from regional_diagnostics import (
    load_sites, sample_ensemble, sample_benchmark,
    stratified_calibration, VARIABLES, VAR_LABEL, LANDCOVER_LABELS,
)

# site coordinates + land cover class (in SDA order)
lon, lat, lc = load_sites()
print(f"{len(lon)} sites, {len(set(lc))} land cover classes")

# sample the 100-member downscaled ensemble and the benchmark at each site,
# once per variable (this is the slow step: it opens the per-member map tiles)
ens_cache = {}
for var, (map_var, bglob, bvar, scale) in VARIABLES.items():
    ens_cache[var] = (sample_ensemble(map_var, lon, lat),
                      sample_benchmark(bglob, bvar, scale, lon, lat))
    print(f"sampled {var}: ensemble {ens_cache[var][0].shape}, obs {ens_cache[var][1].shape}")


## The sites

The SDA sites span North America from the subtropics to the high Arctic. Coloured
by land cover class, the biome distribution follows the expected geography:
tundra in the far north, shrubland through the western drylands, temperate
forests in the east, and grassland and cropland across the plains. This is a
useful check on the land cover labels, which were inferred from the geography and
the biomass and LAI signature of each class.

In [ ]:
from regional_figures import map_sites_by_landcover
map_sites_by_landcover(lon, lat, lc)
from IPython.display import Image
Image("figures/fig_sites_by_landcover.png")


## Calibration by land cover class

For each land cover class we compute the spread-to-error ratio (near 1 for a
well-calibrated ensemble, well below 1 for an overconfident one) and the fraction
of observations that fall inside the ensemble 90 percent band (near 0.90 when
calibrated). We report a domain-wide reference first, then the per-class values.

In [ ]:
for var in VARIABLES:
    members, obs = ens_cache[var]
    dm = ec.spread_skill(members, obs); dc = ec.coverage(members, obs, interval=0.9)
    print(f"\n{VAR_LABEL[var]}")
    print(f"  {'land cover class':40s} {'n':>5s} {'ratio':>7s} {'cov90':>7s}")
    print(f"  {'DOMAIN (all sites)':40s} {members.shape[1]:5d} {dm['ratio']:7.3f} {dc['coverage']:7.3f}")
    for row in stratified_calibration(members, obs, lc, LANDCOVER_LABELS):
        lab = f"{row['group']} {row['label']}"[:40]
        if np.isnan(row['ratio']):
            print(f"  {lab:40s} {row['n']:5d}   (too few)")
        else:
            print(f"  {lab:40s} {row['n']:5d} {row['ratio']:7.3f} {row['cov90']:7.3f}")


In [ ]:
from regional_figures import bars_by_group
lc_rows = {var: stratified_calibration(*ens_cache[var], lc, LANDCOVER_LABELS)
           for var in VARIABLES}
bars_by_group(lc_rows, "Calibration by land cover class", "fig_landcover_bars.png")
Image("figures/fig_landcover_bars.png")


**Reading the land cover breakdown.** The ensemble is overconfident in every
biome and for every variable: no spread-to-error ratio approaches 1, and coverage
is far below 0.90 throughout. The overconfidence is systematically **worst in the
high-carbon forest classes** (temperate, boreal, and tropical forest), where the
observed biomass essentially never falls inside the 90 percent band, and least
severe in the low-carbon classes (grassland, savanna, shrubland, tundra). The
model is most overconfident where the carbon is.

## Calibration by ecoregion

Land cover captures biome type but not geography. We now assign each site an
EPA/CEC North American ecoregion by point-in-polygon, using the same ecoregion
polygons and projection as PEcAn's `EPA_ecoregion_finder`, and group the
diagnostics by Level 1 ecoregion (15 regions across the continent).

In [ ]:
from ecoregion_join import assign_ecoregions, isnull

eco = assign_ecoregions(lon, lat)
l1 = eco["L1"]
matched = int((~isnull(l1)).sum())
print(f"assigned {matched} of {len(lon)} sites to a Level 1 ecoregion "
      f"({len(set(l1[~isnull(l1)]))} regions)")

# build integer groups from region names for the stratification engine
uniq = sorted(set(l1[~isnull(l1)]))
name_to_id = {n: i for i, n in enumerate(uniq)}
labels = {i: n for n, i in name_to_id.items()}
groups = np.array([name_to_id.get(n, -1) for n in l1])

for var in VARIABLES:
    print(f"\n{VAR_LABEL[var]} by L1 ecoregion")
    print(f"  {'ecoregion':40s} {'n':>5s} {'ratio':>7s} {'cov90':>7s}")
    rows = sorted(stratified_calibration(*ens_cache[var], groups, labels),
                  key=lambda r: -r['n'])
    for row in rows:
        if row['n'] < 30: continue
        print(f"  {row['label'][:40]:40s} {row['n']:5d} {row['ratio']:7.3f} {row['cov90']:7.3f}")


In [ ]:
eco_rows = {}
for var in VARIABLES:
    rows = sorted(stratified_calibration(*ens_cache[var], groups, labels), key=lambda r: -r['n'])
    eco_rows[var] = [r for r in rows if r['n'] >= 30]
bars_by_group(eco_rows, "Calibration by L1 ecoregion", "fig_ecoregion_bars.png")
Image("figures/fig_ecoregion_bars.png")


### Mapping the calibration

Shading each ecoregion by its calibration makes the spatial pattern immediate.
For biomass, the whole continent is deep in the overconfident (red) range, with
the forested cores the most extreme. The coverage map tells the same story: the
observed biomass falls outside the 90 percent band almost everywhere, with the
arid Southwest a partial exception.

In [ ]:
import geopandas as gpd
from regional_figures import compute_region_metrics, draw_choropleth, ECO_L1, RATIO_CMAP, COV_CMAP

dissolved = gpd.read_file(ECO_L1).to_crs("EPSG:4326").dissolve(by="NA_L1NAME")
for var in VARIABLES:
    rm = compute_region_metrics(l1, ens_cache, var)
    draw_choropleth(dissolved, rm, "ratio", RATIO_CMAP, 0.0, 1.0,
                    f"{VAR_LABEL[var]}: spread / error by ecoregion (red = overconfident)",
                    f"fig_ecoregion_map_{var}_ratio.png", "spread / error")
    draw_choropleth(dissolved, rm, "cov90", COV_CMAP, 0.0, 0.9,
                    f"{VAR_LABEL[var]}: 90% coverage by ecoregion (red = obs outside)",
                    f"fig_ecoregion_map_{var}_cov.png", "90% coverage")
print("choropleth maps written to figures/")


In [ ]:
Image("figures/fig_ecoregion_map_biomass_ratio.png")

In [ ]:
Image("figures/fig_ecoregion_map_biomass_cov.png")

In [ ]:
Image("figures/fig_ecoregion_map_lai_cov.png")

**Reading the ecoregion breakdown.** The ecoregion view confirms and refines
the land cover result across the full continent, now including the boreal and
Arctic north. Overconfidence is universal, and the forest-versus-open-land
gradient is clear: the most overconfident regions are the forested ones (Eastern
Temperate Forests, Northwestern Forested Mountains, Marine West Coast Forest,
Northern Forests), while the drylands (North American Deserts) and, for some
variables, the tundra are relatively less overconfident. For LAI specifically,
the arid Southwest is the one place the ensemble brackets the observation well,
its coverage approaching the expected 0.90. A notable soil carbon exception is
the Hudson Plain, a vast peatland region where the ensemble is among the most
overconfident, consistent with the difficulty of representing large, uncertain
peatland carbon stocks.

## Findings

Across three independent groupings, domain-wide, by land cover, and by ecoregion,
the picture is consistent:

- **The PEcAn SDA ensemble is overconfident everywhere.** For biomass, soil
  carbon, and LAI, in every biome and every North American ecoregion, the spread
  is too narrow for the error and the observation routinely falls outside the
  ensemble 90 percent range.
- **The overconfidence is worst where carbon stocks are highest and most
  variable**, the forests, and least severe in low-carbon drylands and tundra.
  This is the clearest signal for where the reanalysis most needs improved
  uncertainty representation.
- **There is real regional variation.** For LAI the ensemble is reasonably
  calibrated in the arid Southwest, and for soil carbon the peatland Hudson Plain
  stands out as especially overconfident. The single continent-wide number hides
  this structure, which is exactly why the regional breakdown is useful.

These regional diagnostics reuse the same calibration tool as the domain-wide
assessment, and are built to accept any benchmark, so future benchmark additions
plug into the same regional breakdown.
